## Section 10: Tips & Best Practices for Colab GPU Training

### ⚡ GPU Optimization Tips

1. **Increase Batch Size**: GPU can handle larger batches, reducing training time
   - Typical: 256-512 for RNN models
   - Experiment based on GPU memory (monitor with nvidia-smi)

2. **Use Mixed Precision** (if training custom models):
   ```python
   from torch.cuda.amp import autocast, GradScaler
   scaler = GradScaler()
   with autocast():
       loss = model(input)
   ```

3. **Pipeline Parallelism**: If model > GPU memory, split layers across GPU

4. **Memory Management**:
   - Use `torch.cuda.empty_cache()` between training runs
   - Monitor GPU memory with `torch.cuda.memory_allocated()`

### 🔄 Colab Session Management

- **Session Timeout**: Colab sessions disconnect after inactivity (~90 mins for free tier)
- **Save Frequently**: Save models to Google Drive every epoch/iteration
- **Restore from Checkpoint**: Load best model before final evaluation

### 📊 Performance Expectations

- **Training Speed**: Typically 5-10x faster than CPU
- **GPU Memory**: T4 has 16GB, A100 has 40GB
- **Free Tier Limits**: ~12 hours continuous use

### 🚀 Advanced: Custom Training Loop with GPU

```python
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

for epoch in range(25):
    model.train()
    for batch in train_loader:
        X, y = batch
        X, y = X.to(device), y.to(device)  # Move to GPU
        
        logits = model(X)
        loss = criterion(logits, y)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    # Save checkpoint
    torch.save(model.state_dict(), f'checkpoint_epoch_{epoch}.pt')
```

### 🎯 Quick Command Reference

```bash
# Monitor GPU
!nvidia-smi

# Check GPU availability
!python -c "import torch; print(torch.cuda.is_available())"

# Run training with timeout
!timeout 3600 python models/train.py

# Download files from Colab
from google.colab import files
files.download('models/logistic_model.pkl')
```

---

**✅ You're now ready to run the complete credit risk prediction system on Google Colab with GPU acceleration!**

In [ ]:
# Optional: Run the API with ngrok for external access
# Install ngrok (get your authtoken from https://dashboard.ngrok.com)

!pip install -q pyngrok

from pyngrok import ngrok

# Set ngrok authtoken (replace with your token)
# ngrok.set_auth_token("YOUR_NGROK_AUTH_TOKEN")

# Start the API in background and expose it with ngrok
import subprocess
import threading
import time

def run_api():
    """Run FastAPI in background"""
    subprocess.run(['uvicorn', 'api.main:app', '--host', '0.0.0.0', '--port', '8000'], 
                   capture_output=True)

# Start API in background thread
# api_thread = threading.Thread(target=run_api, daemon=True)
# api_thread.start()
# time.sleep(5)

# Expose with ngrok
# public_url = ngrok.connect(8000)
# print(f"✅ API is live at: {public_url}")

print("⚠️  To run the API:")
print("1. Get your ngrok authtoken from: https://dashboard.ngrok.com")
print("2. Uncomment the code above and replace 'YOUR_NGROK_AUTH_TOKEN'")
print("3. Then you can make predictions via API!")

# Example API call once running:
# curl -X POST "http://localhost:8000/predict?model=logistic" \
#      -H "Content-Type: application/json" \
#      -d '{"income": 75000, "loan_amount": 15000, "credit_score": 720}'

## Section 9: Optional - Run API with GPU Support

Deploy the FastAPI application with GPU-optimized model inference.

In [ ]:
# Save results to Google Drive (if mounted)
import shutil
import os

# Copy all results to Google Drive
def save_to_drive(source_dir, drive_dest):
    """Save results to Google Drive"""
    try:
        # Create destination folder if it doesn't exist
        os.makedirs(drive_dest, exist_ok=True)
        
        # Copy models
        models_dest = os.path.join(drive_dest, 'models')
        if os.path.exists('models'):
            shutil.copytree('models', models_dest, dirs_exist_ok=True)
            print(f"✅ Models saved to: {models_dest}")
        
        # Copy reports
        csv_files = [f for f in os.listdir('.') if f.endswith('.csv')]
        for csv_file in csv_files:
            shutil.copy(csv_file, os.path.join(drive_dest, csv_file))
            print(f"✅ Report saved: {csv_file}")
        
        # Copy visualizations
        png_files = [f for f in os.listdir('.') if f.endswith('.png')]
        for png_file in png_files:
            shutil.copy(png_file, os.path.join(drive_dest, png_file))
            print(f"✅ Visualization saved: {png_file}")
            
    except Exception as e:
        print(f"❌ Error saving to Drive: {e}")

# Save to Google Drive
drive_destination = '/content/drive/My Drive/Credit_Risk_Results'
save_to_drive('.', drive_destination)

print("\n" + "="*80)
print("📁 All results saved!")
print("="*80)

## Section 8: Save and Download Results

Save trained models, evaluation reports, and visualizations for later use.

In [ ]:
# Monitor GPU performance with detailed stats
import subprocess

def monitor_gpu():
    """Display detailed GPU statistics"""
    print("=" * 80)
    print("GPU PERFORMANCE MONITORING")
    print("=" * 80)
    
    # Full nvidia-smi output
    result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    print(result.stdout)
    
    # Memory usage summary
    result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.used,memory.total,temperature.gpu,utilization.gpu', 
                           '--format=csv,noheader'], capture_output=True, text=True)
    lines = result.stdout.strip().split('\n')
    print("\n📊 SUMMARY:")
    for line in lines:
        print(f"  {line}")

monitor_gpu()

# Optional: Create a GPU monitoring loop during training
def continuous_gpu_monitor(duration_seconds=60, interval_seconds=5):
    """Monitor GPU continuously for specified duration"""
    import time
    print(f"\n📈 Monitoring GPU for {duration_seconds} seconds (every {interval_seconds}s)...")
    
    start = time.time()
    while time.time() - start < duration_seconds:
        result = subprocess.run(['nvidia-smi', '--query-gpu=memory.used,memory.total,utilization.gpu,temperature.gpu',
                               '--format=csv,noheader'], capture_output=True, text=True)
        stats = result.stdout.strip()
        timestamp = time.strftime('%H:%M:%S')
        print(f"[{timestamp}] {stats}")
        time.sleep(interval_seconds)

# Use this to monitor during longer training runs
# continuous_gpu_monitor(300, 10)  # Monitor for 5 minutes, update every 10 seconds

## Section 7: Monitor GPU Performance During Training

Real-time GPU monitoring to track memory usage, temperature, and utilization.

In [ ]:
# Run the evaluation pipeline
# This generates fairness metrics, robustness tests, and visualizations

import time

start_time = time.time()

# Run the evaluation script as a module
!python -m utils.evaluation --nrows 100000 --report_path evaluation_report_colab.csv

elapsed_time = time.time() - start_time
print(f"\n✅ Evaluation completed in {elapsed_time:.2f} seconds")

# List generated outputs
print("\n📊 Generated outputs:")
!ls -lh | grep -E "\.csv|\.png"

## Section 6: Run Evaluation Pipeline with Fairness & Robustness Analysis

This section runs the complete evaluation pipeline including fairness metrics and robustness tests.

In [ ]:
# Run the complete training pipeline
# This will train both Logistic Regression and RNN models on GPU

import time

start_time = time.time()

# Run the training script
!python models/train.py

elapsed_time = time.time() - start_time
print(f"\n✅ Training completed in {elapsed_time:.2f} seconds")

# Verify saved models
!ls -lh models/

## Section 5: Run Model Training with GPU

This section trains both Logistic Regression and RNN models using GPU acceleration.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Navigate to your project directory (adjust path as needed)
import os
project_path = '/content/drive/My Drive/Credit_Risk_Loan_Default_Prediction'
os.chdir(project_path)

# List files to verify
!ls -la

**Option B: Mount Google Drive and Access Project Files**

In [ ]:
# Clone your project from GitHub (replace with your repo URL)
!git clone https://github.com/YOUR_USERNAME/Credit_Risk_Loan_Default_Prediction.git
%cd Credit_Risk_Loan_Default_Prediction

# Verify project structure
!ls -la

## Section 4: Setup Project from GitHub or Google Drive

**Option A: Clone from GitHub**

In [ ]:
# Install required packages
!pip install -q numpy pandas scikit-learn matplotlib seaborn plotly joblib

# PyTorch is pre-installed in Colab, but ensure it's GPU-enabled
!pip install -q torch torchvision torchaudio

# For FastAPI and Streamlit (optional, if running API/Dashboard)
!pip install -q fastapi uvicorn streamlit

print("✅ All dependencies installed successfully!")

## Section 3: Install Project Dependencies

In [ ]:
# Check PyTorch GPU availability
import torch

print("PyTorch GPU Check:")
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"Device Count: {torch.cuda.device_count()}")
print(f"Current Device: {torch.cuda.current_device()}")
print(f"Device Name: {torch.cuda.get_device_name(0)}")

# Create a test tensor on GPU
test_tensor = torch.randn(1000, 1000).cuda()
print(f"\nSuccessfully created tensor on GPU: {test_tensor.device}")

In [ ]:
# Check GPU availability
!nvidia-smi

## Section 2: Check GPU Availability

## Section 1: Enable GPU in Google Colab

**Steps to enable GPU:**

1. Click `Runtime` in the menu bar
2. Select `Change runtime type`
3. In the dropdown for "Hardware accelerator", choose `GPU` (preferably `T4` or `A100`)
4. Click `Save`

Your Colab runtime will restart with GPU enabled. You'll see a notification confirming the GPU is allocated.

# Running Credit Risk Prediction Project on Google Colab with GPU

This notebook guides you through running the complete credit risk prediction system on Google Colab with GPU acceleration for faster training.

**Prerequisites:**
- Google Colab account
- Access to Google Drive (for storing data and models)
- GitHub access (optional, for cloning your project repo)